# Inverse Predictor

This notebook trains the PINN model on the dataset, saves the model weights and scalers, and then performs an inverse prediction loop. For the inverse prediction, we freeze the PINN's weights and optimize the input widths using gradient descent to match a target set of performance metrics.

In [4]:
import os
import random
import time
import warnings
import joblib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---------------------------
# 1. LOAD DATA (For Sampling Test)
# ---------------------------
file_path = '../Data/FINAL_4CLASSES.csv'
df = pd.read_csv(file_path, encoding='utf-8', engine='python')

column_mapping = {
    'Gain(db)': 'Gain(dB)', 'Gain': 'Gain(dB)', 'gain': 'Gain(dB)',
    'Bandwidth': 'Bandwidth(Hz)', 'bandwidth': 'Bandwidth(Hz)',
    'GBW': 'GBW(MHz)', 'gbw': 'GBW(MHz)',
    'Power': 'Power(uW)', 'power': 'Power(uW)',
    'PM': 'PM(degree)', 'PhaseMargin': 'PM(degree)',
    'GM': 'GM(dB)',
    'PSRR': 'PSRR(dB)',
    'SlewRate': 'SlewRate (V/us)', 'SlewRate(V/µs)': 'SlewRate (V/us)',
    'CMRR': 'CMRR(dB)', 'class': 'Class', 'CLASS': 'Class'
}
df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns}, inplace=True)

df['Idc(uA)'] = 130.0
df['Length(um)'] = 0.18
df['CL(pF)'] = 10.0
df['CC(pF)'] = 55.0

FEATURE_COLUMNS = [
    'Temperature(°)', 'W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)',
    'Idc(uA)', 'Length(um)', 'CC(pF)', 'CL(pF)'
]
REGRESSION_TARGETS = [
    'Gain(dB)', 'Bandwidth(Hz)', 'GBW(MHz)', 'Power(uW)', 'PM(degree)',
    'GM(dB)', 'PSRR(dB)', 'SlewRate (V/us)', 'CMRR(dB)'
]
CLASSIFICATION_TARGET = 'Class'

# ---------------------------
# 2. LOAD SCALERS AND MODEL
# ---------------------------
scaler_X = joblib.load('scaler_X.pkl')
scaler_y_reg = joblib.load('scaler_y_reg.pkl')
le = joblib.load('label_encoder.pkl')
n_classes = len(le.classes_)
print("Loaded saved scalers.")

class PINN(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_reg_outputs, n_classes, dropout_rate):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.regression_head = nn.Linear(prev_dim, n_reg_outputs)
        self.classification_head = nn.Linear(prev_dim, n_classes)

    def forward(self, x):
        shared = self.backbone(x)
        return self.regression_head(shared), self.classification_head(shared)

pinn = PINN(
    input_dim=len(FEATURE_COLUMNS),
    hidden_dims=[128, 128, 128, 128],
    n_reg_outputs=len(REGRESSION_TARGETS),
    n_classes=n_classes,
    dropout_rate=0.047,
).to(device)

pinn.load_state_dict(torch.load('pinn_model.pth', map_location=device))
pinn.eval()
print("Loaded saved PINN model weights.")


Using device: cpu
Loaded saved scalers.
Loaded saved PINN model weights.


## Provide Target Performance Metrics & Operating Conditions

Fill out the expected performance metrics below. We also setup the fixed operating conditions you provided.

In [5]:
# Select 3 random samples from the dataset
test_samples = df.sample(n=3, random_state=42)

# Ensure model is in eval mode so its weights stay fixed
pinn.eval()
for param in pinn.parameters():
    param.requires_grad = False

width_names = ['W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)']
width_indices = [FEATURE_COLUMNS.index(col) for col in width_names]

for idx, sample_row in test_samples.iterrows():
    print(f"\n{'='*60}")
    print(f"TESTING SAMPLE ROW INDEX: {idx}")
    print(f"{'='*60}")

    # TARGET PERFORMANCE METRICS from dataset
    target_metrics = {col: sample_row[col] for col in REGRESSION_TARGETS}

    # FIXED OPERATING CONDITIONS from dataset
    operating_conditions = {
        'Temperature(°)': sample_row['Temperature(°)'],
        'Idc(uA)': sample_row['Idc(uA)'],
        'Length(um)': sample_row['Length(um)'],
        'CC(pF)': sample_row['CC(pF)'],
        'CL(pF)': sample_row['CL(pF)']
    }

    # Setup the target tensor & scale it using scaler_y_reg
    target_array = np.array([[target_metrics[col] for col in REGRESSION_TARGETS]])
    target_scaled = scaler_y_reg.transform(target_array)
    target_tensor = torch.tensor(target_scaled, dtype=torch.float32).to(device)

    # =========================================================
    # OPTIMIZATION SETUP (With Softplus & Mean Initialization)
    # =========================================================
    fixed_indices = [FEATURE_COLUMNS.index(col) for col in operating_conditions.keys()]

    # Extract dataset mean (mu) and std for widths to initialize and scale correctly
    mu_widths = torch.tensor(scaler_X.mean_[width_indices], dtype=torch.float32, device=device)
    std_widths = torch.tensor(scaler_X.scale_[width_indices], dtype=torch.float32, device=device)

    # Initialize raw parameter at mu_widths (similar to app.py)
    # The true physical width will be softplus(inv_widths_raw)
    inv_widths_raw = mu_widths.clone().detach().requires_grad_(True)

    # Calculate fixed conditional input scalars
    fake_dict = {**operating_conditions, **{k: 0 for k in width_names}}
    initial_features_array = np.array([[fake_dict[col] for col in FEATURE_COLUMNS]])
    initial_scaled = scaler_X.transform(initial_features_array)
    fixed_scaled_vals = torch.tensor(initial_scaled[0, fixed_indices], dtype=torch.float32, device=device)

    # Inverse Optimizer configuration
    inv_epochs = 1500
    inv_optimizer = optim.Adam([inv_widths_raw], lr=0.1)  
    loss_fn = nn.MSELoss()

    print("Running inverse prediction to find optimal widths...\n")
    for epoch in range(1, inv_epochs + 1):
        inv_optimizer.zero_grad()
        
        # 1. Constrain physical width to be POSITIVE using softplus
        widths_phys = F.softplus(inv_widths_raw)
        
        # 2. Scale back the widths internally before feeding to PINN
        widths_scaled = (widths_phys - mu_widths) / std_widths
        
        full_scaled_input = torch.zeros((1, len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
        full_scaled_input[0, width_indices] = widths_scaled
        full_scaled_input[0, fixed_indices] = fixed_scaled_vals
        
        pred_reg, _ = pinn(full_scaled_input)
        
        loss = loss_fn(pred_reg, target_tensor)
        loss.backward()
        inv_optimizer.step()
        
        if epoch % 500 == 0 or epoch == inv_epochs:
            print(f"Inverse Search Epoch {epoch:04d}/{inv_epochs} | MSE Loss (Scaled): {loss.item():.6f}")

    # =========================================================
    # EXTRACT AND VALIDATE RESULTS
    # =========================================================
    with torch.no_grad():
        final_widths_phys = F.softplus(inv_widths_raw).cpu().numpy()
        predicted_widths = {col: final_widths_phys[i] for i, col in enumerate(width_names)}
        
        final_widths_scaled = (F.softplus(inv_widths_raw) - mu_widths) / std_widths
        optimal_input_scaled = torch.zeros((1, len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
        optimal_input_scaled[0, width_indices] = final_widths_scaled
        optimal_input_scaled[0, fixed_indices] = fixed_scaled_vals

    print("\n=============================================")
    print("OPTIMAL INVERSE SIZING RESULTS vs GROUND TRUTH:")
    for k, v in predicted_widths.items():
        true_v = sample_row[k]
        diff_pct = abs(v - true_v) / (true_v + 1e-8) * 100
        print(f" > {k}: Pred={v:.4f} | True={true_v:.4f} | Diff={diff_pct:.2f}%")
    print("=============================================")

    pinn.eval()
    with torch.no_grad():
        pred_reg_eval, _ = pinn(optimal_input_scaled)
        pred_reg_unscaled = scaler_y_reg.inverse_transform(pred_reg_eval.cpu().numpy())
        
    print("\nPREDICTED PERFORMANCE WITH THESE WIDTHS vs TARGET:")
    for i, col in enumerate(REGRESSION_TARGETS):
        target_val = target_metrics[col]
        pred_val = pred_reg_unscaled[0, i]
        error_pct = abs(pred_val - target_val) / (abs(target_val) + 1e-8) * 100
        print(f" > {col}: Target={target_val:.2f} | Predicted={pred_val:.2f} | Diff={error_pct:.2f}%")



TESTING SAMPLE ROW INDEX: 8756
Running inverse prediction to find optimal widths...

Inverse Search Epoch 0500/1500 | MSE Loss (Scaled): 0.042544
Inverse Search Epoch 1000/1500 | MSE Loss (Scaled): 0.023617
Inverse Search Epoch 1500/1500 | MSE Loss (Scaled): 0.005501

OPTIMAL INVERSE SIZING RESULTS vs GROUND TRUTH:
 > W12(um): Pred=0.1628 | True=8.0000 | Diff=97.96%
 > W34(um): Pred=21.3065 | True=30.0000 | Diff=28.98%
 > W58(um): Pred=44.6474 | True=59.9000 | Diff=25.46%
 > W6(um): Pred=14.7669 | True=30.0000 | Diff=50.78%
 > W7(um): Pred=0.8962 | True=30.0000 | Diff=97.01%

PREDICTED PERFORMANCE WITH THESE WIDTHS vs TARGET:
 > Gain(dB): Target=50.15 | Predicted=50.84 | Diff=1.37%
 > Bandwidth(Hz): Target=26213.20 | Predicted=21412.89 | Diff=18.31%
 > GBW(MHz): Target=31.49 | Predicted=32.34 | Diff=2.69%
 > Power(uW): Target=2146.07 | Predicted=2156.55 | Diff=0.49%
 > PM(degree): Target=49.39 | Predicted=50.19 | Diff=1.62%
 > GM(dB): Target=4.87 | Predicted=5.03 | Diff=3.23%
 > PSRR(